# What the capstone agent does to escalate

This notebook runs the **real agent** — `build_complaint_harness` from `agentlab.capstone` — on one complaint and draws straight from the recorded trajectory. It does not re-implement or hand-call any tool; every value shown is what the agent logged.

The complaint is case-016: *"I am outraged. My overdraft fee is unfair and I demand a refund."* The agent's fixed workflow is `classify → extract_facts → search_policy → flag_regulatory → draft / escalate`, and it escalates when `flag_regulatory` returns `escalate=True`.

Running the agent loads its models (classifier, extractor, RAG, flagger, guard), so the run cell is slow. **Run:** *Run ▶ Run All Cells*.

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd()))   # kg_sphere.py / kg_process.py sit here
import kg_sphere as K
import kg_process as P

ROOT = next((c for c in (Path.cwd(), Path.cwd() / 'code',
                         Path.cwd().parent, Path.cwd().parent / 'code')
             if (c / 'data' / 'gms_regulatory_store').exists()), Path.cwd())
STORE = str(ROOT / 'data' / 'gms_regulatory_store')
kg = K.load_gms_store(STORE, source='v')   # the store's entities + triples
print(f'regulatory store: {len(kg.labels)} entities, {len(kg.triples)} triples')

## Run the agent and read its trajectory

`build_complaint_harness` assembles the shipped agent (the five tools, the policy gates and the hash-chained audit log). We run it on case-016 and flatten the trajectory it recorded. Each printed line is a step the agent took and the output that step logged — including `flag_regulatory`'s `flags`, `escalate` and `severity_paths`, and the terminal `escalate` action.

In [ ]:
from forgeloop.agents.capstone import build_complaint_harness
from forgeloop.agents.core import Budget, BudgetTracker, TaskSpec

cases = json.loads((ROOT / 'data' / 'eval_cases' / 'cases.json').read_text())
msg = next(c['message'] for c in cases if c['id'] == 'case-016')
harness, registry = build_complaint_harness(policies_dir=ROOT / 'data' / 'policies')
traj = harness.run(TaskSpec(goal='handle complaint', inputs={'message': msg}),
                   max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))

records = P.records_from_trajectory(traj)
print('final status:', traj.final_state.status)
for r in records:
    if r['kind'] == 'tool_call':
        print(f"  [{r['i']}] {r['tool']:18} -> {r['output']}")
    else:
        print(f"  [{r['i']}] {r['kind'].upper():18} -> {r.get('reason')}")

### The workflow the agent executed

Each box is a step the agent took; the panel under it is that step's **recorded output**, verbatim from the trajectory. `extract_facts` shows the grounded `(product, issue)` and the bound query it generated (`(overdraft, has_fee_amount, ?)`); `flag_regulatory` shows the flags it fired and the severity walk it returned; the terminal `ESCALATE` shows the reason the agent gave.

In [ ]:
fig = P.process_figure(records, title='case-016 · agent trajectory (recorded)')
fig.write_html('kg_agent_trajectory.html', include_plotlyjs=True, full_html=True)
fig

### The stored triplets it walked to escalate

`flag_regulatory` returned `severity_paths = [{flag: UDAAP, severity: high, action: escalate}]`. Those name real store triplets — `udaap has_severity high`, `high has_action escalate` — drawn here on the embedding sphere with their true direction (head → tail), plus the flag's name and statute to identify the regulation. The subgraph is built only from the paths the tool reported.

In [ ]:
sub = P.escalation_subgraph(kg, records)
print(f'{len(sub.labels)} entities, {len(sub.triples)} triples')
for h, r, t in sub.triples:
    print(f'  {h}  {r}  {t}')
K.visualize(sub, title='case-016 · UDAAP escalation path',
            arrow_size=0.10, out_html='kg_escalation_path.html')

## The full regulatory graph (context)

The whole store the guard is built over — every relation, colored.

In [ ]:
ARROW_SIZE = 0.08
K.visualize(kg, title='Regulatory GMS Store (full)', arrow_size=ARROW_SIZE)

## Semantic view: drop the alias edges

`has_alias` is the largest, densest relation; dropping it leaves the regulatory structure the guard reasons over.

In [ ]:
keep = [r for r in kg.relations if r != 'has_alias']
K.visualize_store(STORE, source='v', relations=keep,
                  title='Regulatory GMS Store (semantic relations)',
                  arrow_size=ARROW_SIZE)